# Работа с пропусками данных — простыми словами

Учебный ноутбук: что делать с **NaN** / пустыми ячейками, чтобы модель могла учиться **честно** (без утечки) и по возможности **умно**.

Разбираем:
- не трогать / удалить / заполнить;
- **SimpleImputer** (mean, median, most_frequent, constant);
- **индикатор пропуска**;
- **fit / transform** и data leakage;
- **Pipeline + кросс-валидация**;
- **KNNImputer** и **IterativeImputer**;
- пропуски во **временных рядах** (ffill, bfill, interpolate).

**Библиотеки:** `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

Запускайте ячейки **сверху вниз**.

---

## План

1. [Словарь](#dict)
2. [Какие стратегии бывают](#strategies)
3. [Не трогать пропуски](#leave)
4. [Удалить строки](#drop)
5. [SimpleImputer — простая импутация](#simple)
6. [fit / transform и утечка данных](#leakage)
7. [Pipeline + K-Fold](#pipeline)
8. [Индикатор пропуска](#indicator)
9. [Категориальные признаки](#cat)
10. [KNNImputer](#knn)
11. [IterativeImputer и «ручная» регрессионная импутация](#iter)
12. [Временные ряды: ffill / bfill / interpolate](#ts)
13. [Шпаргалка](#итог)


<a id="dict"></a>
## 1. Словарь

| Термин | Простыми словами |
|--------|------------------|
| **Пропуск / NaN** | «Данных нет» в ячейке |
| **Импутация** | **Заполнить** пропуск каким-то значением |
| **MCAR** | Пропуск «совсем случайный» (факт пропуска сам по себе ничего не значит) |
| **MAR / MNAR** | Пропуск связан с другими полями / с самим скрытым значением (факт пропуска **может** быть сигналом) |
| **`fit`** | «Запомнить правила по **train**» (среднее, соседей, модели…) |
| **`transform`** | «Применить **уже запомненное**» к train/valid/test |
| **Утечка (leakage)** | В train «затекла» информация из test/valid |
| **Pipeline** | Конвейер: импутер + модель, чтобы CV был честным |
| **Индикатор пропуска** | Новый столбец: «здесь **был** NaN» (0/1) |

> **Уточнение:** типы MCAR/MAR/MNAR — полезная рамка, но на практике точный тип часто **неизвестен**.  
> Если есть подозрение, что «не указал доход» ≠ случайность — полезны **constant / Missing** и **индикатор**.


<a id="strategies"></a>
## 2. Какие стратегии бывают (карта)

| Стратегия | Идея |
|-----------|------|
| **Не трогать** | Модель сама умеет NaN (бустинги, некоторые деревья) |
| **Удалить строки / столбец** | Выкинуть «дыры» |
| **Simple impute** | Одно значение на столбец (mean/median/mode/const) |
| **+ индикатор** | Заполнили **и** сказали модели «тут был пропуск» |
| **KNN impute** | Заполнить по **похожим** строкам |
| **Iterative / model impute** | Предсказать пропуск **моделью** по другим признакам |
| **Временные** | ffill / bfill / interpolate |

Ниже — от простого к сложному.


<a id="leave"></a>
## 3. Не трогать пропуски

Некоторые модели **умеют** работать с NaN **сами**:

1. **XGBoost**  
2. **LightGBM**  
3. **CatBoost** (есть своя обработка пропусков)  
4. Часть реализаций деревьев  

**Идея:** алгоритм сам решает, куда «отправить» пропуск при разбиении.

**Когда ок:** вы сознательно используете такой алгоритм и понимаете его политику пропусков.

**Когда не ок:** линейная регрессия, обычный SVM, kNN-классификатор, нейросеть «в лоб» — им нужен **заполненный** числовой вход (или отдельная схема).

> «Не трогать» ≠ «игнорировать проблему качества данных».  
> Долю пропусков и причину всё равно стоит **посмотреть**.


<a id="drop"></a>
## 4. Удалить строки с пропусками

### Когда применяют

1. Пропусков **очень мало** (ориентир: единицы процентов, не догма).  
2. Строка **не уникально ценна**.  
3. После удаления данных **всё ещё достаточно**.

### Плюсы

1. Просто.  
2. **Не** вносит искусственных чисел.  
3. Не «размазывает» одно среднее по всем дырам.

### Минусы

Можно выкинуть много строк и **сместить** выборку (пропуски часто **не** случайны: не заполнили форму → другой тип клиента).

### Код (pandas)

```python
df = df.dropna()                      # хотя бы один NaN в строке
df = df.dropna(subset=["salary"])     # NaN только в salary критичен
df = df.dropna(axis=1, thresh=...)    # иногда удаляют почти пустые столбцы
```


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "name": ["Ann", "Bob", "Cara", "Dan", "Eve"],
    "age": [25, np.nan, 31, 28, np.nan],
    "salary": [50, 60, np.nan, 55, 70],
    "city": ["A", "B", "A", np.nan, "B"],
})
print("Исходные данные:")
print(df)
print()
print("dropna() — выкинули строки с любым NaN:")
print(df.dropna())
print()
print("dropna(subset=['salary']) — только если нет salary:")
print(df.dropna(subset=["salary"]))
print()
print(f"Было строк: {len(df)}, после dropna: {len(df.dropna())} "
      f"({100*len(df.dropna())/len(df):.0f}% осталось)")


<a id="simple"></a>
## 5. SimpleImputer — простая импутация

**Импутация** = заменить пропуск значением.

```python
from sklearn.impute import SimpleImputer
```

### Стратегии

| strategy | Для чего | Что подставляет |
|----------|----------|-----------------|
| `"mean"` | числа | среднее |
| `"median"` | числа | медиану |
| `"most_frequent"` | чаще **категории** (и числа) | моду |
| `"constant"` | что угодно | `fill_value` |

### Mean

**Когда:** число, распределение **примерно симметричное**, **нет** сильных выбросов.  
**Минус:** среднее **ломается** от выбросов.

### Median — самый частый выбор для чисел

**Когда:** выбросы, асимметрия (доход, цены…).  
**Плюс:** устойчивее среднего.

### most_frequent

Цвет: Красный, Синий, Красный, **NaN**, Красный → чаще **Красный**.  
Практически стандарт для категорий (альтернатива — константа `"Missing"`).

### constant

Когда **сам факт** пропуска важен: не указал доход, образование, анализ.

```python
SimpleImputer(strategy="constant", fill_value="Unknown")
SimpleImputer(strategy="constant", fill_value=-1)  # для чисел — осторожно: -1 может стать «выбросом»
```

### Главная мысль

SimpleImputer **не угадывает** «настоящее» значение.  
Он считает **одно** число/метку **на столбец по train** и клеит его во **все** пропуски этого столбца.  
Зато **быстро** и стабильно — хорошая **точка старта**.


In [ ]:
from sklearn.impute import SimpleImputer

# Числовой столбец с выбросом
x = np.array([[10.0], [12.0], [11.0], [1000.0], [np.nan], [13.0]])
print("Данные (есть выброс 1000 и NaN):")
print(x.ravel())

for strategy in ["mean", "median"]:
    imp = SimpleImputer(strategy=strategy)
    filled = imp.fit_transform(x)
    print(f"strategy={strategy!r}: statistics_={imp.statistics_[0]:.2f}, "
          f"NaN → {filled[-2,0]:.2f}")

# Категории
cities = np.array([["A"], ["B"], ["A"], [np.nan], ["A"], ["C"]], dtype=object)
imp_cat = SimpleImputer(strategy="most_frequent")
print("\nКатегории:", cities.ravel())
print("most_frequent →", imp_cat.fit_transform(cities).ravel())
print("запомненная мода:", imp_cat.statistics_)

imp_unk = SimpleImputer(strategy="constant", fill_value="Unknown")
print("constant Unknown →", imp_unk.fit_transform(cities).ravel())


### Как работают `fit` и `transform`

| Шаг | Что происходит |
|-----|----------------|
| `imputer.fit(X_train)` | Считает mean/median/mode **только по train** и **запоминает**. **Ещё не** заполняет. |
| `imputer.transform(X_*)` | Подставляет **запомненное** вместо NaN |

**Правильно:**

```python
imputer.fit(X_train)
X_train = imputer.transform(X_train)
X_test  = imputer.transform(X_test)
# или:
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)   # без fit на test!
```

**Никогда:** `imputer.fit(X_test)` или `fit` на всём датасете до честного split — это путь к **утечке**.


<a id="leakage"></a>
## 6. Утечка данных при импутации

### Неправильно: `fit` на всех данных

Train возраст: 20, 25, 30, 35 → среднее **27.5**  
Test: 60, 70, 80, 90 → среднее **75**  
`fit` на train+test → среднее **~51**  

В train подставили число, в котором **уже есть** «запах» test.  
Модель косвенно видела статистику экзамена.

### Правильно

1. Разделить train / test.  
2. `fit` **только** на train → запомнили 27.5.  
3. `transform` train **и** test → везде 27.5, даже если test «на самом деле» другой.

Так и в проде: пришёл новый клиент с `age=NaN` — подставляем статистику **с обучения**, а не «среднее по всем будущим клиентам».

### Аналогия

Нельзя сказать: «Подождите, я сначала посмотрю средний возраст тех, кто придёт **через месяц**».  
Есть только то, что знали **на момент обучения**.


In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
# train — «молодые», test — «старше» (утрированно)
age_train = np.array([20, 25, 30, 35, 22, 28, np.nan, 33], dtype=float)
age_test  = np.array([60, 70, 80, 90, np.nan, 75], dtype=float)

# --- НЕПРАВИЛЬНО: fit на train+test ---
all_ages = np.concatenate([age_train, age_test]).reshape(-1, 1)
bad = SimpleImputer(strategy="mean")
bad.fit(all_ages)
print(f"Неправильно, statistics_ (с test) = {bad.statistics_[0]:.2f}")

# --- ПРАВИЛЬНО ---
good = SimpleImputer(strategy="mean")
good.fit(age_train.reshape(-1, 1))
print(f"Правильно, statistics_ (только train) = {good.statistics_[0]:.2f}")

print("transform test (подставили train-mean):",
      good.transform(age_test.reshape(-1, 1)).ravel())
print("Среднее test без NaN (модель его НЕ должна «знать»):",
      np.nanmean(age_test))


<a id="pipeline"></a>
## 7. Pipeline + кросс-валидация

Чтобы в каждом фолде импутер **не** видел valid:

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression()),
])

cross_val_score(pipeline, X_train, y_train, cv=KFold(5, shuffle=True, random_state=42),
                scoring="neg_mean_squared_error")
```

**В каждом фолде автоматически:**

1. `fit` imputer на **train_fold**  
2. `transform` train_fold и **valid_fold**  
3. `fit` model на заполненном train_fold  
4. оценка на valid_fold  

После выбора пайплайна:

```python
pipeline.fit(X_train, y_train)   # imputer.fit на ВСЁМ X_train
pipeline.predict(X_test)         # transform test уже запомненными статистиками
```


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=120, n_features=3, noise=10.0, random_state=0)
# искусственные пропуски
X_miss = X.copy()
mask = np.random.default_rng(1).random(X.shape) < 0.15
X_miss[mask] = np.nan

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression()),
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_miss, y, cv=kf, scoring="neg_mean_squared_error")
print("CV MSE (median impute внутри Pipeline):", np.round(-scores, 2))
print("mean MSE:", (-scores).mean().round(2))
print("→ в каждом фолде median считался заново только по train_fold.")


<a id="indicator"></a>
## 8. Индикатор пропуска (Missing Indicator)

### Идея

Не только заполнить NaN, но и добавить признак:

> «Здесь **изначально** был пропуск» (1/0).

### Когда полезно

Факт отсутствия **несёт смысл**: не указал доход, образование, не сдал анализ  
(банки, медицина, страхование, скоринг).

### Плюсы / минусы

**+** просто, дёшево, часто чуть лучше качество.  
**−** если пропуски полностью случайны (MCAR) — пользы может не быть;  
добавляются столбцы (по одному на каждый столбец с NaN).

### Код

```python
SimpleImputer(strategy="median", add_indicator=True)
# или отдельно: from sklearn.impute import MissingIndicator
```

На практике чаще **`add_indicator=True`**.

### Схема

```text
пропуск → SimpleImputer заполнил значение
        → добавил столбец «был пропуск»
        → модель видит ОБА сигнала
```

Индикатор **не заменяет** импутацию — работает **вместе** с ней.


In [ ]:
from sklearn.impute import SimpleImputer

X_demo = np.array([
    [25.0, 50.0],
    [np.nan, 60.0],
    [40.0, np.nan],
    [30.0, 55.0],
], dtype=float)

imp = SimpleImputer(strategy="median", add_indicator=True)
X_out = imp.fit_transform(X_demo)
print("Исходное X:")
print(X_demo)
print("\nПосле impute + indicator (последние столбцы — флаги пропусков):")
print(X_out)
print("statistics_ (медианы):", imp.statistics_)


<a id="cat"></a>
## 9. Категориальные признаки: что делают в ~95% проектов

| Метод | Когда |
|-------|--------|
| **`"Missing"` / `"Unknown"`** (constant) | Факт пропуска информативен — **часто №1** |
| **`most_frequent`** | Хотим «типичную» категорию |
| **Удалить строки** | Дыр мало |
| **Модельная импутация** (классификатор) | Редко: важный признак, много пропусков, хорошо предсказывается |

В sklearn **нет** универсального «CategoricalIterativeImputer»: слишком много вариантов (бинар/мультикласс, дисбаланс, выбор модели).  
Поэтому модельное заполнение категорий обычно **ручное** и редкое.

Правила leakage **те же**: учить заполнитель только на train / внутри фолда.

```python
SimpleImputer(strategy="most_frequent")
SimpleImputer(strategy="constant", fill_value="Missing")
```


<a id="knn"></a>
## 10. KNNImputer — заполнение по соседям

### Идея

Не одно среднее на всех, а: **найди похожие строки** и возьми у них значение.

Пример (вес = NaN):

| age | height | weight |
|-----|--------|--------|
| 25 | 175 | 70 |
| 27 | 178 | 74 |
| 60 | 165 | 85 |
| 26 | 176 | **NaN** → соседи ~25–27 → вес ≈ (70+74)/2 = **72** |
| 58 | 166 | **NaN** → сосед ~60 → вес ≈ **85** |

Разные строки → **разные** заполнения.

### Что делает `fit` / `transform`

| | SimpleImputer | KNNImputer |
|--|---------------|------------|
| `fit` | запоминает **1 число** на столбец | запоминает **строки train** (база соседей) |
| `transform` | подставляет это число | для каждой строки ищет соседей **в train** и усредняет |

Valid/test: соседи **только из train** (если `fit` был на train) → утечки нет.

**Неправильно:** `fit` на valid или на всём X до K-Fold так, что valid-строки попадают в базу соседей.

### `n_neighbors`

Гиперпараметр (часто 3–10). Подбирают через CV **внутри** Pipeline.


In [ ]:
from sklearn.impute import KNNImputer

X_knn = np.array([
    [25.0, 175.0, 70.0],
    [27.0, 178.0, 74.0],
    [60.0, 165.0, 85.0],
    [26.0, 176.0, np.nan],
    [58.0, 166.0, np.nan],
])
print("До:")
print(X_knn)

imp = KNNImputer(n_neighbors=2)
print("\nПосле KNNImputer(n_neighbors=2):")
print(np.round(imp.fit_transform(X_knn), 2))
print("→ у молодых вес ближе к 70–74, у старших — к 85 (не одно среднее).")


### Расстояние: `nan_euclidean`

Обычное евклидово **не умеет** NaN.  
KNNImputer использует **`nan_euclidean_distances`**:

- признаки, где у **хотя бы одной** из двух строк NaN, в пару **не входят**;
- расстояние **масштабируют**, чтобы строки с кучей пропусков не казались «всегда близкими».

Упрощённо (как в типичном примере с $p$ признаками и $m$ общими):

$$
d = \sqrt{\frac{p}{m}\sum_{i\in\text{общие}}(x_i-y_i)^2}
$$

Пример: разности 1 и 2, третий признак не сравнили ($p=3$, $m=2$):

- «голое» $\sqrt{1+4}\approx 2.24$  
- nan-евклидово $\sqrt{(3/2)\cdot 5}\approx 2.74$ — **штраф** за неполное сравнение.

### Масштаб признаков — боль KNN

Возраст 18–80 и доход 30 000–1 000 000: доход **задавит** расстояние.

**Плохо:** `KNNImputer` → потом `StandardScaler` (соседи уже найдены «криво»).

**Лучше** (в современных sklearn `StandardScaler` **игнорирует NaN** при `fit` и **оставляет** NaN при `transform`):

```python
Pipeline([
    ("scaler", StandardScaler()),      # сначала масштаб
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", LinearRegression()),
])
```

Тогда соседи ищутся в **нормализованном** пространстве.  
Для смеси число/категории KNN обычно только на **числовом** блоке (`ColumnTransformer`).

### Ограничения KNNImputer

1. Медленно на больших данных ($\sim$ сравнение с train).  
2. Шум и много признаков портят «похожесть».  
3. Если пропуск в **самом важном** поле — соседи ищутся **без него** → могут быть «не те».  
4. Категории «в лоб» не кодирует.

Поэтому в серьёзных проектах часто: **IterativeImputer**, своя модель, или KNN только после аккуратной подготовки.


In [ ]:
from sklearn.metrics.pairwise import nan_euclidean_distances
from sklearn.preprocessing import StandardScaler

# Проверка формулы nan_euclidean
a = np.array([[0.0, 0.0, np.nan]])
b = np.array([[1.0, 2.0, 5.0]])
d = nan_euclidean_distances(a, b)[0, 0]
print(f"nan_euclidean ≈ {d:.3f} (ожидаем ≈2.739)")
print(f"без штрафа p/m: {np.sqrt(1+4):.3f}")

# Масштаб: StandardScaler + NaN
X_s = np.array([[1.0, 1000.0], [np.nan, 1100.0], [3.0, 900.0]])
print("\nStandardScaler с NaN:")
print(StandardScaler().fit_transform(X_s))

# Pipeline: scale → knn → model
X_big, y_big = make_regression(n_samples=80, n_features=4, noise=8.0, random_state=2)
Xb = X_big.copy()
Xb[np.random.default_rng(3).random(Xb.shape) < 0.12] = np.nan

pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", LinearRegression()),
])
sc = cross_val_score(pipe_knn, Xb, y_big, cv=KFold(5, shuffle=True, random_state=0),
                     scoring="neg_mean_squared_error")
print("\nCV MSE (StandardScaler → KNNImputer → LR):", round((-sc).mean(), 2))


<a id="iter"></a>
## 11. IterativeImputer и регрессионная импутация

### Регрессионная импутация (идея «вручную»)

«Для столбца с NaN обучи **регрессию** по остальным признакам и предскажи пропуски.»

- **Один** столбец с дырами — понятно и часто достаточно.  
- **Несколько** столбцов — сами решаете порядок, число проходов → муторно.

**IterativeImputer** = эта идея **автоматом** и **итеративно**.

### Как работает IterativeImputer

1. **Грубо** заполняет все NaN (часто средним/медианой).  
2. Для каждого столбца по очереди:  
   - считает его **целью**;  
   - учит модель $X_j = f(\text{остальные столбцы})$;  
   - обновляет пропуски в $X_j$.  
3. Повторяет **раунды** (`max_iter`, по умолчанию до сходимости по `tol`).

Почему **Iterative**: возраст → рост → вес → снова возраст… с всё более хорошими значениями.

### Что запоминает `fit`

| Метод | После `fit` |
|-------|-------------|
| SimpleImputer | одно значение / столбец |
| KNNImputer | строки train |
| **IterativeImputer** | **обученные модели** на столбцы |

`transform` на новых данных **не** переучивает — использует сохранённые модели.

### Под капотом

По умолчанию estimator ≈ **BayesianRidge** (линейная регрессия с регуляризацией).  
Можно: `RandomForestRegressor`, `ExtraTreesRegressor`, дерево…

```python
from sklearn.experimental import enable_iterative_imputer  # обязательно!
from sklearn.impute import IterativeImputer

IterativeImputer(max_iter=10, random_state=42)
```

Ориентирован на **числа**. Категории — другие подходы.

### Важный нюанс обучения столбца

Когда учат, например, **вес** по росту и возрасту, в обучение регрессора берут строки, где вес **изначально известен** (иначе «сам себя» учит на своих же грубых подстановках — замкнутый круг).  
Строки с пропусками в **других** полях (уже грубо/итеративно заполненных) **используются** как входы — компромисс, чтобы не выкидывать почти все данные.

### `tol`

Остановка, если изменения заполненных значений стали малы (в sklearn критерий **относительный**, не «просто 0.001 кг»).  
Можно задать `max_iter` и `tol` явно.

### Плюсы / минусы

**+** учитывает **зависимости** между признаками; часто качественнее Simple/KNN.  
**−** медленнее, тяжелее, может переобучаться, сложнее объяснить.

### В Pipeline / K-Fold

Так же, как Simple/KNN: imputer **внутри** Pipeline → `fit` только на train_fold.


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# Маленький пример: вес ~ связан с ростом
X_it = np.array([
    [25.0, 175.0, 70.0],
    [30.0, 180.0, 78.0],
    [40.0, 170.0, 72.0],
    [28.0, 178.0, np.nan],
    [np.nan, 172.0, 68.0],
    [35.0, np.nan, 80.0],
], dtype=float)

print("До IterativeImputer:")
print(X_it)

it = IterativeImputer(max_iter=20, random_state=0)
X_filled = it.fit_transform(X_it)
print("\nПосле:")
print(np.round(X_filled, 2))

# Сравнение Simple vs KNN vs Iterative на синтетике с зависимостью
rng = np.random.default_rng(5)
n = 100
a = rng.normal(30, 5, size=n)
h = 150 + 0.8 * a + rng.normal(0, 2, size=n)
w = 0.4 * a + 0.3 * h + rng.normal(0, 1.5, size=n)
X_full = np.column_stack([a, h, w])
# прячем 20% значений
Xm = X_full.copy()
m = rng.random(Xm.shape) < 0.2
Xm[m] = np.nan

def mae_impute(imputer):
    # impute, сравниваем только бывшие NaN
    Xhat = imputer.fit_transform(Xm)
    return np.mean(np.abs(Xhat[m] - X_full[m]))

print("\nMAE на скрытых ячейках (меньше = лучше восстановление):")
print("  Simple median:", round(mae_impute(SimpleImputer(strategy="median")), 3))
print("  KNN(5):       ", round(mae_impute(KNNImputer(n_neighbors=5)), 3))
print("  Iterative:    ", round(mae_impute(IterativeImputer(max_iter=15, random_state=0)), 3))


### Три метода одной фразой

| Метод | Фраза |
|-------|--------|
| **SimpleImputer** | «Всем одна и та же заплата-заглушка.» |
| **KNNImputer** | «Как у **похожих** людей.» |
| **IterativeImputer** | «Как предскажет **модель** по другим полям.» |


<a id="ts"></a>
## 12. Временные ряды: ffill / bfill / interpolate

Когда строки идут **по времени**, «среднее по столбцу» может **сломать** хронологию.  
Часто смотрят на **соседей по времени**.

### Forward fill (`ffill`)

Пропуск ← **последнее известное** значение «сверху» (вчерашняя цена/температура).

```python
df["temp"] = df["temp"].ffill()
```

**Когда:** акции, датчики, курсы, остатки на складе — «если сегодня не пришло, часто как вчера».

**Минус:** длинная дыра → долгое «залипание» одного значения.

### Backward fill (`bfill`)

Пропуск ← **следующее** известное. Реже в ML.

```python
df["temp"] = df["temp"].bfill()
```

### Интерполяция

Между 20 и 24 на соседних днях → середина **22** (линейно по умолчанию).

```python
df["temp"] = df["temp"].interpolate()
```

**Когда:** плавные ряды (температура, давление, часть финансовых).  
**Минус:** при резких скачках «придумает» несуществовавшее значение.

> **Утечка во времени:** `bfill` / интерполяция, смотрящая в **будущее**, для прогноза «вперёд» часто **нечестны**.  
> Для честного forecasting обычно **ffill** / модели с лагом, без подглядывания вперёд.


In [ ]:
# Временной ряд
ts = pd.DataFrame({
    "day": [1, 2, 3, 4, 5, 6],
    "temp": [20.0, np.nan, np.nan, 24.0, np.nan, 25.0],
})
print("Исходный ряд:")
print(ts)

ts_ff = ts.copy()
ts_ff["ffill"] = ts_ff["temp"].ffill()
ts_ff["bfill"] = ts["temp"].bfill()
ts_ff["interp"] = ts["temp"].interpolate()
print("\nСравнение методов:")
print(ts_ff)

# Устаревший вариант pandas: df.fillna(method="ffill") — лучше .ffill()
print("\n(Используйте Series.ffill() / bfill() / interpolate(), не fillna(method=...).)")


<a id="итог"></a>
## 13. Самое главное + шпаргалка

### Выбор метода

| Ситуация | Старт |
|----------|--------|
| Дыр очень мало | `dropna` (осторожно со смещением) |
| Числа, быстро и просто | **median** SimpleImputer |
| Числа + выбросы | median (не mean) |
| Категории | `most_frequent` или **`"Missing"`** |
| Факт пропуска важен | constant / Missing + **`add_indicator=True`** |
| Есть связи между числами | **IterativeImputer** или ручная регрессия |
| Похожие объекты, умеренный размер | **KNNImputer** (+ scale **до** KNN) |
| Временной ряд | **ffill** / interpolate (без утечки из будущего) |
| Модель = бустинг | иногда можно **не** импутить |

### Железные правила

1. **`fit` только на train** (или train_fold).  
2. Test/valid только **`transform`**.  
3. В CV импутер — **внутри Pipeline**.  
4. Simple = одно значение на столбец; KNN = соседи; Iterative = модели.  
5. Индикатор **дополняет** импутацию, не заменяет.  
6. Прод: новый клиент с NaN → те же правила, что выучили на train.

### Мини-глоссарий

| | |
|--|--|
| **Imputation** | заполнение |
| **statistics_** | что запомнил SimpleImputer |
| **nan_euclidean** | расстояние KNN с NaN |
| **enable_iterative_imputer** | импорт-флаг sklearn для IterativeImputer |


### Мини-практика

1. На таблице с NaN сравните `mean` vs `median` при выбросе.  
2. Покажите leakage: `fit` на train+test vs только train.  
3. Соберите `Pipeline(SimpleImputer → LinearRegression)` и `cross_val_score`.  
4. Включите `add_indicator=True` и посмотрите форму `transform`.  
5. (Опционально) Сравните MAE восстановления Simple / KNN / Iterative на синтетике.


In [ ]:
# ===== Сводка «всё в одном коротком скрипте» =====
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler

X, y = make_regression(n_samples=100, n_features=5, noise=12.0, random_state=7)
X[np.random.default_rng(7).random(X.shape) < 0.1] = np.nan
cv = KFold(5, shuffle=True, random_state=7)

candidates = {
    "median": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ]),
    "median+ind": Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", LinearRegression()),
    ]),
    "knn+scale": Pipeline([
        ("scaler", StandardScaler()),
        ("imp", KNNImputer(n_neighbors=5)),
        ("model", LinearRegression()),
    ]),
    "iterative": Pipeline([
        ("imp", IterativeImputer(max_iter=10, random_state=0)),
        ("model", LinearRegression()),
    ]),
}

print("CV mean MSE (меньше лучше):")
for name, pipe in candidates.items():
    mse = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_squared_error").mean()
    print(f"  {name:12s}  {mse:.2f}")
print("\nЧисла на вашей машине могут чуть отличаться — важен честный Pipeline, не гонка за 0.01.")


## Что делать дальше

1. Свяжите с ноутбуком **«Валидация…»**: любой imputer = кандидат на leakage, лечится Pipeline.  
2. Свяжите с **метриками**: качество смотрите на valid/test **после** честной импутации.  
3. В проде логируйте долю NaN — резкий рост пропусков = сигнал о поломке данных.

### Главная мысль

> Пропуск — это тоже информация (иногда).  
> Заполнять можно просто или умно, но **всегда**: правила учат на train,  
> а test только **получает** уже выученные правила.

Удачи!
